In [1]:
import numpy as np
import cv2
import joblib
import os
import pandas as pd
import tifffile as tiff
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

print("Imports successful!")

Imports successful!


In [2]:
# --- Paths ---
DATA_PATH = '../data/raw/HyperLeaf2024/HyperLeaf2024/'
IMG_FOLDER = os.path.join(DATA_PATH, "images")
TRAIN_CSV_PATH = os.path.join(DATA_PATH, "train.csv")

# Path to the pipeline we just built in notebook 02
PIPELINE_PATH = '../models/hyperleaf_full_pipeline.joblib'

# Output paths for processed data
PROCESSED_DIR = '../data/processed/'
os.makedirs(PROCESSED_DIR, exist_ok=True)

# --- Constants ---
MODEL_HEIGHT = 128
MODEL_WIDTH = 128

# --- Load Pipeline and Data ---
print("Loading preprocessing pipeline...")
pipeline = joblib.load(PIPELINE_PATH)

try:
    train_labels_df = pd.read_csv(TRAIN_CSV_PATH)
    print(f"Loaded {len(train_labels_df)} records from train.csv")
except FileNotFoundError:
    print(f"ERROR: Cannot find train.csv at: {TRAIN_CSV_PATH}")
    raise # Stop the notebook

Loading preprocessing pipeline...
Loaded 1590 records from train.csv


In [3]:
data = []
labels = []

print("Applying pipeline to all 1590 images...")
# This will be fast because it's not fitting, just transforming
for index, row in tqdm(train_labels_df.iterrows(), total=len(train_labels_df)):
    img_id_int = int(row['ImageId'])
    img_id_str = str(img_id_int).zfill(5)
    img_path = os.path.join(IMG_FOLDER, f"{img_id_str}.tiff")

    try:
        raw_image = tiff.imread(img_path)
        
        # 1. Resize
        resized_image = cv2.resize(raw_image, (MODEL_WIDTH, MODEL_HEIGHT), interpolation=cv2.INTER_LINEAR)
        
        # 2. Reshape
        h, w, bands = resized_image.shape
        reshaped_pixels = resized_image.reshape(-1, bands)
        
        # 3. APPLY PIPELINE (using .transform, NOT .fit)
        processed_pixels = pipeline.transform(reshaped_pixels)
        
        # 4. Reshape back to image
        # Use .astype('float32') to save memory
        processed_image = processed_pixels.reshape(h, w, 3).astype('float32')
        
        data.append(processed_image)
        labels.append(row['Fertilizer']) # Or 'Nutrient' if that's the column

    except Exception as e:
        print(f"Warning: Skipping {img_path}. Error: {e}")

print("Finished processing all images.")

Applying pipeline to all 1590 images...


100%|██████████████████████████████████████████████████████████████████████████████| 1590/1590 [04:29<00:00,  5.89it/s]

Finished processing all images.


In [4]:
print("Converting data to NumPy arrays...")
X = np.array(data)
y = np.array(labels)

print(f"Final X shape: {X.shape}") # Should be (1590, 128, 128, 3)
print(f"Final y shape: {y.shape}")

# --- Process Labels ---
# We use LabelEncoder to turn strings ("N-0", "N-1") into numbers (0, 1)
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

# Save the encoder so your app can map predictions back to names
joblib.dump(encoder, os.path.join(PROCESSED_DIR, 'label_encoder.joblib'))
print(f"Label encoder saved. Classes: {encoder.classes_}")

# --- Split the data ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, 
    test_size=0.2,        # 20% for testing
    random_state=42,      # For reproducible results
    stratify=y_encoded    # Ensures balanced classes in train/test
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

Converting data to NumPy arrays...
Final X shape: (1590, 128, 128, 3)
Final y shape: (1590,)
Label encoder saved. Classes: [0.  0.5 1. ]
X_train shape: (1272, 128, 128, 3)
X_test shape: (318, 128, 128, 3)
y_train shape: (1272,)
y_test shape: (318,)


In [5]:
# --- Save the new, clean data ---
print("Saving final processed data...")
np.save(os.path.join(PROCESSED_DIR, "X_train_v2.npy"), X_train)
np.save(os.path.join(PROCESSED_DIR, "y_train_v2.npy"), y_train)
np.save(os.path.join(PROCESSED_DIR, "X_test_v2.npy"), X_test)
np.save(os.path.join(PROCESSED_DIR, "y_test_v2.npy"), y_test)

print("✅ Data processing and saving complete.")
print(f"All files saved in: {os.path.abspath(PROCESSED_DIR)}")

Saving final processed data...
✅ Data processing and saving complete.
All files saved in: C:\Users\DELL\Mini Project\data\processed
